In [ ]:
# notebook for running all the components of the pipeline for reproducing the baseline paper
import os
import matplotlib.pyplot as plt
import numpy as np
import cv2

try:
    from google.colab import drive
    os.chdir('/content')
    drive.mount('/content/drive', force_remount=True)

    !pip install opencv-python
    !pip install joblib
    os.chdir('/content/drive/MyDrive/3dgs_decomposition')
    !pip install -r intr_decomp_3dgs/requirements.txt
    !pip install intr_decomp_3dgs/diff-gaussian-rasterization  --no-build-isolation -v
    !pip install intr_decomp_3dgs/third_party/simple-knn intr_decomp_3dgs/third_party/fused_ssim --no-build-isolation
except ModuleNotFoundError:
    pass  # running locally, already in project root

In [ ]:
from splat.fit_baseline import align_depth_scale, run_baseline_train, render_test_views
from splat.fit_baseline_gsplat import load_scene, fit_albedo, freeze_albedo, freeze_albedo_shading
import torch

In [ ]:
scenes = [('hypersim', 'ai_001_001'), ('hypersim', 'ai_002_001')]
# already done ('mipnerf360', 'bicycle'), ('mipnerf360', 'garden'), ('mipnerf360', 'room')
albedo_predictors = ['colourful_iid', 'marigold', 'rgbx']

In [ ]:
for dataset, scene in scenes:
  if dataset == 'mipnerf360':
    align_depth_scale(scene_dir=f'data/{dataset}/{scene}', depths_dir=f'depth_anything_v2/depths/{dataset}/{scene}')
  else:
    align_depth_scale(scene_dir=f'data/{dataset}/colmap_format/{scene}/cam_00', depths_dir=f'depth_anything_v2/depths/{dataset}/{scene}')

In [ ]:
# run baseline train and render and evaluate test views
for dataset, scene in scenes:
  for predictor in albedo_predictors:
    output_dir = os.path.abspath(f'baseline_train/{predictor}/{dataset}/{scene}')
    albedos_path = os.path.abspath(f'{predictor}/{dataset}/{scene}/albedos')
    depth_paths = os.path.abspath(f'depth_anything_v2/depths/{dataset}/{scene}')
    if dataset == 'mipnerf360':
      source_path = os.path.abspath(f'data/{dataset}/{scene}/images_8')
      images_folder = 'images_8'
    elif dataset == 'hypersim':
      source_path = os.path.abspath(f'data/{dataset}/colmap_format/{scene}/cam_00/images')
      images_folder = 'images'
    else:
      print('Invalid Dataset Name')
      raise KeyboardInterrupt

    os.makedirs(output_dir, exist_ok=True)

    run_gsplat_pipeline(
        root=source_path,
        dataset=dataset,
        scene=scene,
        predictor=predictor,
        output_dir='gsplat_train')

    render_test_views(
        source_path=source_path,
        output_dir=output_dir,
        iterations=10000
    )

In [ ]:
def run_gsplat_pipeline(root, dataset, scene, predictor, output_dir="gsplat_train",
                        iters_albedo=10000, iters_shading=3000, iters_residual=3000, seed=0):
    root = os.path.abspath(root)
    source_path, images_folder = scene_paths(root, dataset, scene)
    out_dir = os.path.join(root, output_dir, predictor, dataset, scene)

    os.makedirs(out_dir, exist_ok=True)

    np.random.seed(seed)
    torch.manual_seed(seed)

    data = load_scene(root, dataset, scene, predictor, source_path, images_folder)
    tr, te = data["train"], data["test"]
    W, H, ss = data["width"], data["height"], data["scene_scale"]
    print(f"{predictor}/{dataset}/{scene}: {len(tr['names'])} train / {len(te['names'])} test views, "
          f"{W}x{H}, {len(data['points'])} points, scene_scale={ss:.3f}")

    common = dict(scene=scene, predictor=predictor, ckpt_dir=out_dir)
    fit_albedo(data["points"], tr["viewmats"], tr["intrinsics"], W, H, tr["images"], tr["albedos"],
               tr["depths"], iters=iters_albedo, scene_scale=ss, **common)
    freeze_albedo(tr["viewmats"], tr["intrinsics"], W, H, tr["images"],
                  iters=iters_shading, scene_scale=ss, **common)
    freeze_albedo_shading(tr["viewmats"], tr["intrinsics"], W, H, tr["images"],
                          iters=iters_residual, scene_scale=ss, **common)

    # result = evaluate_test_views(te, W, H, scene, predictor, out_dir, out_dir)
    del data, tr, te
    torch.cuda.empty_cache()
    # return result